# ReportLens — OCR fine-tuning (TrOCR) on Colab

Fine-tunes `microsoft/trocr-small-printed` on **synthetic lab-report lines** and reports
Character Error Rate (CER). Run the cells top-to-bottom.

**Before you start:** set the GPU runtime →  *Runtime → Change runtime type → Hardware
accelerator: **T4 GPU** → Save*.

Lower CER is better (0.0 = perfect). Aim for well under ~0.1. Training takes roughly
30–45 min on a T4 with the default settings. The engineering story here is the whole
pipeline: synthetic data → classical line segmentation → a fine-tuned recogniser, with
the same crop preprocessing used in training and inference to avoid train/serve skew.

## 1. Check the GPU is on

In [ ]:
!nvidia-smi

## 2. Clone the repo (always pulls the latest code)

Pulls in both `ocr_engine` and the `data_synthesis` package that generates the training
data. If the repo is already there from an earlier run, it is **reset to the latest
`main`** — otherwise a stale checkout would silently keep running old code.

The cell prints the commit it's running, so you can confirm you have the newest fixes.

In [ ]:
import os
%cd /content
# Robust clone: plain Python if/else with single-line shell commands (a multi-line bash
# `if` in a `!` cell is fragile in Colab). If the repo exists from an earlier run it is
# reset to the latest main so pushed fixes always take effect.
if os.path.isdir('reportlens'):
    !cd reportlens && git fetch -q origin && git reset -q --hard origin/main
else:
    !git clone -q https://github.com/Satwiksingh123/reportlens-backend-.git reportlens
%cd /content/reportlens
print('--- running commit ---')
!git log --oneline -1

## 3. Install the training dependencies (heavy — ~2-3 min)

Pins `transformers<5` (v5 changed tokenizer loading and breaks TrOCR) and installs
`sentencepiece`/`protobuf`, which TrOCR needs to build its tokenizer.

If pip prints a message about restarting the runtime, do *Runtime → Restart session*, then
re-run this cell and continue. Training itself runs in a fresh process, so it picks up the
right versions either way.

In [ ]:
!pip install -q -e "services/ocr_engine[train]"
!pip install -q pillow  # data_synthesis rendering
# TrOCR builds its fast tokenizer by converting the slow one, which needs sentencepiece
# + protobuf. Install explicitly so the conversion can't fail on a fresh Colab runtime.
!pip install -q sentencepiece protobuf

# Check versions in a SUBPROCESS. A plain `import transformers` here would report whatever
# this kernel loaded earlier in the session, not what is actually installed on disk.
!python -c "import transformers, sentencepiece; print('transformers', transformers.__version__); print('sentencepiece OK')"

## 4. Train (auto-restores from Drive if you've trained before)

Colab disconnects the runtime after idle time (or a long training run), which wipes the
local disk — so a model trained in an earlier session disappears even though nothing in
the notebook changed. This cell now **mounts your Google Drive** and checks
`MyDrive/reportlens_ocr/trocr-lab` first:

- **Found** -> restores it in seconds, no retraining.
- **Not found** -> generates synthetic reports, crops each labelled word, fine-tunes TrOCR,
  prints the CER, then **saves the result to Drive** for next time.

The first run will ask you to authorize Drive access — click through the Google prompt.
Training itself takes roughly 25–40 min on a T4 when it does run.

In [ ]:
import os
# Self-bootstrap: clone the repo if a prior cell didn't (so this cell works even if run on
# its own). Then install deps and train.
%cd /content
if not os.path.isdir('reportlens'):
    !git clone -q https://github.com/Satwiksingh123/reportlens-backend-.git reportlens
%cd /content/reportlens
!pip install -q -e "services/ocr_engine[train]" pillow sentencepiece protobuf
%cd /content/reportlens/services/ocr_engine

# Colab disconnects/recycles the VM after idle time or a long run, wiping the local disk -
# any model trained earlier in a DIFFERENT session is gone even though this notebook looks
# unchanged. Mount Drive and restore from there if a previous run already saved one, so a
# reconnect never forces a full ~30-40 min retrain.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_MODEL_DIR = '/content/drive/MyDrive/reportlens_ocr/trocr-lab'

if os.path.isdir(DRIVE_MODEL_DIR):
    print(f'[restore] found a saved model at {DRIVE_MODEL_DIR}, restoring (skipping training)')
    os.makedirs('artifacts', exist_ok=True)
    import shutil
    if os.path.isdir('artifacts/trocr-lab'):
        shutil.rmtree('artifacts/trocr-lab')
    shutil.copytree(DRIVE_MODEL_DIR, 'artifacts/trocr-lab')
else:
    print('[train] no saved model found on Drive - training now')
    # Word-level training on trocr-base-printed. Words give many crops per report, so a
    # modest report count is plenty.
    !python -m ocr_engine.train_trocr --num-reports 250 --epochs 3 --batch-size 8 \
        --out artifacts/trocr-lab
    print(f'[save] copying the trained model to Drive ({DRIVE_MODEL_DIR}) so a future '
          'disconnect does not require retraining')
    import shutil
    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    if os.path.isdir(DRIVE_MODEL_DIR):
        shutil.rmtree(DRIVE_MODEL_DIR)
    shutil.copytree('artifacts/trocr-lab', DRIVE_MODEL_DIR)
    print('[save] done')

## 5. Quick sanity check on one synthetic page

**If you already trained a model earlier in this session, you do NOT need to retrain** —
this cell only changed how the model *decodes* (a beam-search length bias was cutting
outputs short, e.g. "0.3" -> "0."). Just re-run cell 2 (pulls the fix, keeps
`artifacts/trocr-lab` since it's untracked) and then this cell.

Runs the **full pipeline** (segmentation + your fine-tuned recogniser) on a freshly
generated report and prints the ground truth next to the OCR output.

In [ ]:
import os
# Self-bootstrap so this works even if run on its own. Runs in a subprocess so it uses the
# installed library versions, not whatever this kernel imported earlier.
%cd /content
if not os.path.isdir('reportlens'):
    !git clone -q https://github.com/Satwiksingh123/reportlens-backend-.git reportlens
%cd /content/reportlens/services/ocr_engine

# If cell 4 hasn't run in THIS session (fresh runtime), restore from Drive rather than
# failing with "model not found" - avoids re-running the training cell just to get the
# model back after a disconnect.
if not os.path.isdir('artifacts/trocr-lab'):
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/reportlens_ocr/trocr-lab'
    if os.path.isdir(drive_dir):
        import shutil
        os.makedirs('artifacts', exist_ok=True)
        shutil.copytree(drive_dir, 'artifacts/trocr-lab')
        print(f'[restore] copied model from {drive_dir}')
    else:
        print('[!] no local model and none saved on Drive yet - run cell 4 (Train) first.')

!python -m ocr_engine.sanity_check --model-dir artifacts/trocr-lab

## 6. Download the fine-tuned model
Zips the model folder so you can save it (or upload it to the API host later). It is too
large for git — keep it as a release asset / on Drive, not in the repo.

In [ ]:
%cd /content/reportlens/services/ocr_engine
!cd artifacts && zip -qr trocr-lab.zip trocr-lab
from google.colab import files
files.download('artifacts/trocr-lab.zip')

### (Optional) Save to Google Drive instead
```python
from google.colab import drive; drive.mount('/content/drive')
!cp artifacts/trocr-lab.zip /content/drive/MyDrive/
```